# Reality Bites

Part 8 built a RAG system that worked beautifully -- but it worked on a book: one document, written by one author, clean structure throughout. Most real corpora don't look like that. They're scattered across different systems, in inconsistent formats, full of internal jargon and administrative shorthand.

This part builds a bot that helps a student choose a TAF (Thématique d'Approfondissement) at IMT Atlantique, mixing RAG -- grounded in real course descriptions, not invented ones -- with genuine agentic decision-making. The corpus itself is real too: 279 actual TAF fiches, and once we index them with the exact same code that worked so well on the book, retrieval quality drops noticeably. Diagnosing *why*, and fixing it, is where most of the real work in a RAG system actually lives.

In [ ]:
# Program 1: basic setup

import os
from dotenv import load_dotenv

load_dotenv(override=True)

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "documents"))

## The corpus: real TAF fiches from PASS


Unlike the book, nobody hands us a single tidy PDF for this. IMT Atlantique's actual TAF catalogue lives in **PASS**, behind a Shibboleth SSO login that only staff and students can reach -- not something a web-search agent can just find and download.

So this corpus wasn't collected by an agent inside this notebook. It comes from `browse_pass.py`, a small standalone script sitting alongside this notebook, built with **Playwright**: a library that drives a real browser -- clicking, navigating, waiting for pages to load, reading back whatever ends up on screen. Playwright shows up constantly in agentic AI, wired up as a tool an agent decides for itself when and how to use (an MCP browser server, for instance, or Part 6's own tools). Here it's used the opposite way: `browse_pass.py` always does the exact same sequence of clicks, decided in advance, with no LLM anywhere making a choice -- pure algorithmic browser automation, the same distinction Part 6 already drew between a fixed workflow and a real agent, just applied to driving a browser instead of calling tools.

What it actually does: opens a real Chrome window, waits for a human to type their SSO password (that part genuinely can't be scripted), then walks PASS's "Catalogue UE - TAF" page, clicking every course entry, capturing the popup window that opens for each one -- reading its content frame by frame straight out of the page's HTML, since PASS still uses old-school `<frameset>`s -- and saving the result as a `.txt` file. Run once by hand, not as a repeatable notebook cell, for the same SSO reason -- and now, for the server-load reason above, not something to repeat at all.

What it leaves behind is reproducible, though: 279 fiches already sitting in `documents/taf/`. Open one or two yourself before moving on -- they're plain text files, so any editor works.

⚠️ **Do not run `browse_pass.py`.** It's here to be read, not run: PASS is IMT Atlantique's internal course-management system, never built for a whole class launching real browser automation against it at once -- a well-intentioned script clicking through hundreds of course pages simultaneously, multiplied by every student in the room, is exactly the kind of load a small internal server doesn't survive. The 279 fiches it already collected, once, are what you actually work with below.

In [ ]:
# Program 2: the TAF fiches already collected from PASS via browse_pass.py

TAF_DIR = os.path.join(DOCS_DIR, "taf")
taf_files = sorted(f for f in os.listdir(TAF_DIR) if f.endswith(".txt"))
print(f"{len(taf_files)} fiches in documents/taf/\n")

print("A few of them:")
for filename in taf_files[:5]:
    print(f"  {filename}")
print("  ...")

print("\nOne fiche, to see the shape of the data:")
with open(os.path.join(TAF_DIR, taf_files[0])) as f:
    print(f.read()[:600])

Each fiche keeps the metadata `browse_pass.py` captured alongside the content -- which TAF-choice menu it came from (`CHOIX_TAF`), the course title (`TITRE_FICHE`), and the PASS URL it was popped up from -- then the raw page text underneath, frame by frame exactly as PASS rendered it: course code, responsable(s), équipe pédagogique, credits, site, and so on.

`documents/taf/` stays local -- not committed, the same way `.env` isn't -- but it's what every program below actually indexes.

A couple of fiches got captured twice: `browse_pass.py`'s click-everything approach followed a date-picker link that happened to reopen a course popup it had already saved under a different filename. 279 files, 273 distinct UE codes -- worth knowing about before it quietly duplicates a course in an index later.

## Cleaning the fiches through an LLM

PASS imposes the same section headers on every fiche -- Description, Contenus, Prérequis, Résultats d'apprentissages, Compétences, and so on -- but what actually lands in them isn't consistent. `UE coeur de la TAF` is filled in for some courses and blank for others (whichever doesn't apply to that course), responsable lists run from one name to five, and structure imposed by the platform doesn't guarantee the same information sits in the same place from one fiche to the next.

So instead of writing a parser for that structure -- or a chunking fix, like this part used to reach for -- the fix this time is at extraction: hand each raw fiche to an LLM and ask it to reformat it into a clean, consistent record, trying to lose as little real information as possible along the way. Beyond the pedagogical description (what we'll actually embed) and the responsable(s)' names (kept in their own file, next to the UE they belong to), it's worth pulling out while we're already reading the whole fiche: the intended **learning outcomes** ("Résultats d'apprentissages visés" -- what a student should be able to do afterwards, not just what the course covers), the **competency blocks** it develops ("III. Compétences développées dans l'UE" -- a codified accreditation reference, e.g. `BC02-DSC-2`, distinct from the free-text learning outcomes), and the **campus** it's taught on. A structured-output call, not an agent: no tools, no multi-turn decisions, just "read this, give me back these fields" -- the same LangChain chat-model interface Part 8's Program 7 introduced, this time with `.with_structured_output()` instead of a plain `.invoke()`.

One real constraint decides how this runs: Gemini's free tier caps at 15 requests per minute, and there are 273 distinct fiches once duplicates are dropped -- at least 20 minutes of pure rate-limit waiting, with no way around it. That's too slow for a notebook cell you re-run in class, for the same reason `browse_pass.py` isn't one: it just needs to happen once. `clean_fiches.py`, sitting alongside this notebook, does the full batch -- paced under the rate limit, checkpointed to disk after every fiche so an interruption doesn't lose progress -- and caches the result to `documents/taf/cleaned.json`. Below, the extraction runs live on a couple of fiches so you can see it actually work, then we load the full cached result.

In [ ]:
# Program 3: extract pedagogical content and responsables through an LLM -- live, on a couple of fiches

from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# BaseModel (pydantic) defines the *shape* of the answer we want back: each attribute below
# becomes one field in a JSON schema, its type and Field(description=...) telling the model
# exactly what to put there. with_structured_output(FicheExtract) wraps the chat model so it
# no longer returns free text -- the response is forced to match that schema (JSON, under the
# hood), and LangChain parses it straight into a FicheExtract instance, so result.ue_code,
# result.responsables etc. are real Python attributes, not a string we'd have to parse
# ourselves. Same idea as Part 2's @function_tool, which turns a Python function's type hints
# into a schema too -- just applied here to a model's final answer instead of a tool call.
# A BaseModel can also nest another one: Competency below is its own little schema, and
# FicheExtract's competencies field is just "a list of those."
class Competency(BaseModel):
    code: str = Field(description="The competency block code, e.g. BC02-DSC-2")
    description: str = Field(description="What this competency block covers")

class FicheExtract(BaseModel):
    ue_code: str = Field(description="The UE code, e.g. PA-DI-ADEEPL-B")
    title: str = Field(description="The course title, cleaned up -- no numbering, no UE code prefix")
    description: str = Field(description="Clean pedagogical description: what the course covers, its "
                              "content and prerequisites -- prose only, no PASS UI labels, table "
                              "clutter, or administrative metadata like credits or campus.")
    learning_outcome: str = Field(description="The intended learning outcomes ('Résultats "
                                   "d'apprentissages visés' section) -- what a student should be able "
                                   "to do after completing the course, not just what it covers.")
    competencies: List[Competency] = Field(description="Each competency block listed under 'III. "
                                            "Compétences développées dans l'UE' / 'Instanciations des "
                                            "blocs de compétences' -- code and description.")
    campus: str = Field(description="The campus/site the course is taught on (e.g. Brest, Rennes, "
                         "Nantes), from the 'UE proposée sur le site de' field.")
    responsables: List[str] = Field(description="Names of the UE responsable(s), exactly as listed "
                                     "in the Responsable(s) field.")

clean_llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest", google_api_key=os.environ["GOOGLE_API_KEY"])
structured_clean_llm = clean_llm.with_structured_output(FicheExtract)

for filename in taf_files[:2]:
    with open(os.path.join(TAF_DIR, filename)) as f:
        raw_text = f.read()
    # The whole fiche, not a truncated prefix -- 219 of the 279 fiches are longer than 4000
    # characters (up to 13,500), and Compétences/Résultats d'apprentissages usually sit well
    # past that point. Truncating silently drops exactly the fields we're here to extract.
    result = structured_clean_llm.invoke(
        f"Extract the pedagogical content from this course fiche:\n\n{raw_text}")
    print(f"{result.ue_code} -- {result.title} ({result.campus})")
    print(f"  Responsables: {', '.join(result.responsables)}")
    print(f"  Description: {result.description[:200]}...")
    print(f"  Learning outcome: {result.learning_outcome[:200]}...")
    print(f"  Competencies: {', '.join(c.code for c in result.competencies)}\n")

# Just a demo -- these two results aren't saved anywhere. clean_fiches.py makes the exact
# same call for real, checkpointing every result to documents/taf/cleaned.json as it goes.

That's the mechanism -- the same call `clean_fiches.py` makes 273 times, paced and checkpointed, to produce `documents/taf/cleaned.json`. Load the real result, and split it into what gets embedded and what gets kept alongside each UE's name.

In [ ]:
# Program 4: load the cleaned corpus, split into embeddable descriptions and a responsables file

import json

CLEANED_PATH = os.path.join(TAF_DIR, "cleaned.json")
with open(CLEANED_PATH) as f:
    cleaned_fiches = json.load(f)

print(f"{len(cleaned_fiches)} / 273 fiches cleaned")  # clean_fiches.py may still be running

taf_descriptions = [fiche["description"] for fiche in cleaned_fiches]

responsables_by_ue = {fiche["ue_code"]: {"title": fiche["title"], "campus": fiche["campus"],
                                          "responsables": fiche["responsables"]}
                       for fiche in cleaned_fiches}
RESPONSABLES_PATH = os.path.join(TAF_DIR, "responsables.json")
with open(RESPONSABLES_PATH, "w") as f:
    json.dump(responsables_by_ue, f, ensure_ascii=False, indent=2)
print(f"Responsables for {len(responsables_by_ue)} UEs saved to documents/taf/responsables.json\n")

print("One cleaned record, to compare with the raw fiche Program 2 printed:")
sample = cleaned_fiches[0]
print(f"{sample['ue_code']} -- {sample['title']} ({sample['campus']})")
print(f"  Responsables: {', '.join(sample['responsables'])}")
print(f"  Description: {sample['description'][:300]}...")
print(f"  Learning outcome: {sample['learning_outcome'][:300]}...")
print(f"  Competencies: {', '.join(c['code'] for c in sample['competencies'])}")

## Leftover: Part 8's local embedding toolkit

Recreated from Part 8 early in this rebuild, before it was clear whether the TAF pipeline would still need a hand-rolled local embedding model, a fixed-size chunker, or a cosine-similarity `retrieve()`. Nothing above actually calls any of this right now -- parked here until the embedding step (LangChain + Chroma, most likely) makes clear whether it's still needed.

In [ ]:
# Part 8's local embedding toolkit -- kept here in case we still need it, not currently used

import re
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

embed_name = "intfloat/multilingual-e5-small"
print(f"Loading {embed_name}...")
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()
print(f"  loaded: {embed_model.config.num_hidden_layers} layers, "
      f"{embed_model.config.hidden_size}-dimensional embeddings")

def embed(texts, batch_size=32):
    """Same mean-pooling as Part 7/8, plus length-1 normalisation, batched so a few hundred
    passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

def chunk_text(text, size=180, overlap=40):
    """Cut text into overlapping passages of `size` words -- Part 8's fixed-size chunker."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

def is_useful(chunk):
    """Drop chunks that are mostly punctuation and page numbers, not real text."""
    letters = sum(character.isalpha() for character in chunk)
    return letters / max(len(chunk), 1) > 0.6

def retrieve(question, chunks, vectors, top_k=3):
    question_vector = embed_query(question)
    scores = vectors @ question_vector          # one dot product per chunk, in one operation
    best = scores.topk(top_k)
    return [(scores[i].item(), chunks[i]) for i in best.indices.tolist()]

# Self-test: embed a couple of unrelated passages plus one query, and confirm retrieve()
# actually finds the right one -- proof the toolkit works before anything leans on it.
test_chunks = ["The weather in Rennes is often rainy.", "Python is a popular programming language."]
test_vectors = embed_passages(test_chunks)
test_score, test_chunk = retrieve("what language is used for coding?", test_chunks, test_vectors, top_k=1)[0]
print(f"Self-test: {test_score:.3f} similarity, retrieved {test_chunk!r}")
print("Local embedding model and chunking helpers ready.")